# This is a sample Jupyter Notebook of creating data with using LLM

In this notebook we download the data from Hugging Face. Then we generate new data using LLM.

First we load the dependencies

In [13]:
import pandas as pd
from datasets import load_dataset, DownloadMode
import requests
from tqdm import tqdm

Load the Data from Hugging Face

In [14]:
# Add progress bar integration
tqdm.pandas();

# Load the Gen Z slang dataset
dataset = load_dataset("MLBtrio/genz-slang-dataset", download_mode=DownloadMode.REUSE_CACHE_IF_EXISTS)
df = pd.DataFrame(dataset['train'])
df.head(5)

Generating train split: 100%|██████████| 1779/1779 [00:00<00:00, 131242.60 examples/s]


,Slang,Description,Example,Context
0,W,Shorthand for win,"Got the job today, big W!",Typically used in conversations to celebrate s...
1,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.",Often used when referring to a failure or mish...
2,L+ratio,Response to a comment or action on the interne...,Your tweet got 5 likes and 100 replies calling...,Popularized on social media platforms to signi...
3,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to me...
4,Cheugy,Derogatory term for Millennials. Used when mil...,"That phrase is so cheugy, no one says that any...",Used to refer to things that were once popular...


## Add informal text that does contain slang words

Then we create a prompt for telling the LLM what to do with the data.
Rewrite normal English text so the purpose aligns with the original slang meaning. This ensures the refined text captures the same sentiment/intent as the slang.

In [56]:
def normalize_slang(example, slang, description, normal_text=""):
    """Uses the local Mistral API to rewrite slang-laden text into normal English."""
    prompt = f"""You are a language refinement expert. Rewrite the following text to preserve the original intent and sentiment of the slang term '{slang}' (meaning: {description}).

    Requirements:
    - Use informal, conversational English
    - Capture the same emotional tone and purpose as the original slang
    - Output only UTF-8 text characters (no emojis or special symbols)
    - Provide ONLY the rewritten sentence with no explanations, quotes, or parentheses
    - Do not include any meta-commentary about the task

    Original text: {example}

    Rewritten version:"""

    try:
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                "model": "mistral",
                "prompt": prompt,
                "stream": False,
                "temperature": 0.3,
                "top_p": 0.9
            },
            timeout=15
        )

        response.raise_for_status()
        data = response.json()
        result = data.get("response", "").strip()

        return result.split("\n")[0] if result else normal_text

    except requests.exceptions.JSONDecodeError:
        print("Warning: JSON parse error. Skipping this row.")
        return normal_text

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return normal_text


Calling the API for each row in the Genz-slang from MLBtrio. The output would be the informal text generated from the LLM.

In [57]:
# === Apply normalization with progress bar ===
df["Informal_Text"] = df.progress_apply(
    lambda row: extract_slang(
        row["Example"],
        row["Slang"],
        row["Description"]
    ),
    axis=1
)

100%|██████████| 1779/1779 [09:25<00:00,  3.14it/s]


See the new column created in the dataset

In [58]:
df.head(5)


,Slang,Description,Example,Context,Informal_Text,Informal_Text_Without_Slang,Formal_Text
0,W,Shorthand for win,"Got the job today, big W!",Typically used in conversations to celebrate s...,"Nailed it today, big win!","Landed the job today, big success!","Secured the employment position today, signifi..."
1,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.",Often used when referring to a failure or mish...,"Dang it, left my wallet at home – that's a los...","Oops, I left my wallet at home, that's a bummer.",I left my wallet at home; that's a setback.
2,L+ratio,Response to a comment or action on the interne...,Your tweet got 5 likes and 100 replies calling...,Popularized on social media platforms to signi...,"Dang, your tweet only managed five likes but a...",Your tweet garnered 5 likes but attracted a hu...,Your tweet earned five likes but was met with ...
3,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to me...,That meme's fire as hell!,That meme's utterly fantastic!,That meme is remarkably impressive!
4,Cheugy,Derogatory term for Millennials. Used when mil...,"That phrase is so cheugy, no one says that any...",Used to refer to things that were once popular...,"Man, that's totally cringey – nobody's using t...","That's so uncool and outdated, nobody uses tha...",That expression seems unfashionable and outdat...


Save the dataset to a file

In [59]:
df.to_csv("genz_slang.csv", index=False)

In [60]:
df.Informal_Text

0                               Nailed it today, big win!
1       Dang it, left my wallet at home – that's a los...
2       Dang, your tweet only managed five likes but a...
3                               That meme's fire as hell!
4       Man, that's totally cringey – nobody's using t...
                              ...                        
1774                      Time to snooze, catch ya later!
1775    Holy smokes, I can't even believe you pulled o...
1776     School's got a no-nonsense approach to bullying.
1777                           Yo, what's crackin' today?
1778    Man, I was knackered, hit the sack, and was as...
Name: Informal_Text, Length: 1779, dtype: object

## Add informal text that does not contain slang words

In [30]:
def normalize_slang(example, slang, description, normal_text=""):
    """Uses the local Mistral API to rewrite slang-laden text into normal English."""
    prompt = f"""You are a language refinement expert. Rewrite the following text to preserve the original intent and sentiment of the slang term '{slang}' (meaning: {description}).

    Requirements:
    - Use informal, conversational English without slang
    - Capture the same emotional tone and purpose as the original slang
    - Output only UTF-8 text characters (no emojis or special symbols)
    - Provide ONLY the rewritten sentence with no explanations, quotes, or parentheses
    - Do not include any meta-commentary about the task

    Original text: {example}

    Rewritten version:"""

    try:
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                "model": "mistral",
                "prompt": prompt,
                "stream": False,
                "temperature": 0.3,
                "top_p": 0.9
            },
            timeout=15
        )

        response.raise_for_status()
        data = response.json()
        result = data.get("response", "").strip()

        return result.split("\n")[0] if result else normal_text

    except requests.exceptions.JSONDecodeError:
        print("Warning: JSON parse error. Skipping this row.")
        return normal_text

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return normal_text


In [31]:
# === Apply normalization with progress bar ===
df["Informal_Text_Without_Slang"] = df.progress_apply(
    lambda row: extract_slang(
        row["Example"],
        row["Slang"],
        row["Description"]
    ),
    axis=1
)

100%|██████████| 1779/1779 [09:25<00:00,  3.15it/s]


In [32]:
df.head(5)

,Slang,Description,Example,Context,Informal_Text,Informal_Text_Without_Slang
0,W,Shorthand for win,"Got the job today, big W!",Typically used in conversations to celebrate s...,"Scored the job today, major win!","Landed the job today, big success!"
1,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.",Often used when referring to a failure or mish...,"D'aww man, I left my wallet at home – that's a...","Oops, I left my wallet at home, that's a bummer."
2,L+ratio,Response to a comment or action on the interne...,Your tweet got 5 likes and 100 replies calling...,Popularized on social media platforms to signi...,Your tweet garnered 5 likes yet was met with a...,Your tweet garnered 5 likes but attracted a hu...
3,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to me...,That meme is insanely great!,That meme's utterly fantastic!
4,Cheugy,Derogatory term for Millennials. Used when mil...,"That phrase is so cheugy, no one says that any...",Used to refer to things that were once popular...,That line feels so outdated and overly trendy ...,"That's so uncool and outdated, nobody uses tha..."


In [33]:
df.to_csv("genz_slang.csv", index=False)
df.Informal_Text_Without_Slang

0                      Landed the job today, big success!
1        Oops, I left my wallet at home, that's a bummer.
2       Your tweet garnered 5 likes but attracted a hu...
3                          That meme's utterly fantastic!
4       That's so uncool and outdated, nobody uses tha...
                              ...                        
1774                            It's bedtime, good night!
1775            Good grief, I can't believe you did that!
1776         Our school doesn't tolerate bullying at all.
1777                 Hey, what's going on with you today?
1778    Man, I was knackered, hit the sack and was fas...
Name: Informal_Text_Without_Slang, Length: 1779, dtype: object

## Add formal text

In [50]:
def normalize_slang(example, slang, description, normal_text=""):
    """Uses the local Mistral API to rewrite slang-laden text into normal English."""
    prompt = f"""You are a language refinement expert. Rewrite the following text to preserve the original intent and sentiment of the slang term '{slang}' (meaning: {description}).

    Requirements:
    - Use formal, conversational English without slang
    - Capture the same emotional tone and purpose as the original slang
    - Output only UTF-8 text characters (no emojis or special symbols)
    - Provide ONLY the rewritten sentence with no explanations, quotes, or parentheses
    - Do not include any meta-commentary about the task

    Original text: {example}

    Rewritten version:"""

    try:
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                "model": "mistral",
                "prompt": prompt,
                "stream": False,
                "temperature": 0.3,
                "top_p": 0.9
            },
            timeout=15
        )

        response.raise_for_status()
        data = response.json()
        result = data.get("response", "").strip()

        return result.split("\n")[0] if result else normal_text

    except requests.exceptions.JSONDecodeError:
        print("Warning: JSON parse error. Skipping this row.")
        return normal_text

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return normal_text


In [51]:
# === Apply normalization with progress bar ===
df["Formal_Text"] = df.progress_apply(
    lambda row: extract_slang(
        row["Example"],
        row["Slang"],
        row["Description"]
    ),
    axis=1
)

100%|██████████| 1779/1779 [09:41<00:00,  3.06it/s]


In [53]:
df.head(5)

,Slang,Description,Example,Context,Informal_Text,Informal_Text_Without_Slang,Formal_Text
0,W,Shorthand for win,"Got the job today, big W!",Typically used in conversations to celebrate s...,"Scored the job today, major win!","Landed the job today, big success!","Secured the employment position today, signifi..."
1,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.",Often used when referring to a failure or mish...,"D'aww man, I left my wallet at home – that's a...","Oops, I left my wallet at home, that's a bummer.",I left my wallet at home; that's a setback.
2,L+ratio,Response to a comment or action on the interne...,Your tweet got 5 likes and 100 replies calling...,Popularized on social media platforms to signi...,Your tweet garnered 5 likes yet was met with a...,Your tweet garnered 5 likes but attracted a hu...,Your tweet earned five likes but was met with ...
3,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to me...,That meme is insanely great!,That meme's utterly fantastic!,That meme is remarkably impressive!
4,Cheugy,Derogatory term for Millennials. Used when mil...,"That phrase is so cheugy, no one says that any...",Used to refer to things that were once popular...,That line feels so outdated and overly trendy ...,"That's so uncool and outdated, nobody uses tha...",That expression seems unfashionable and outdat...


In [54]:
df.to_csv("genz_slang.csv", index=False)
df.Formal_Text

0       Secured the employment position today, signifi...
1             I left my wallet at home; that's a setback.
2       Your tweet earned five likes but was met with ...
3                     That meme is remarkably impressive!
4       That expression seems unfashionable and outdat...
                              ...                        
1774                         It's bedtime now, goodnight!
1775    Good heavens, I am utterly astonished by what ...
1776    Our school maintains a strict zero-tolerance p...
1777                  Greetings, how are you doing today?
1778    I was utterly exhausted; I retired to bed almo...
Name: Formal_Text, Length: 1779, dtype: object

## Looking into data set from Programmer-RD-AI for genz-slang
First we will load the data

In [38]:
# Load the Gen Z slang dataset
dataset_pairs = load_dataset("Programmer-RD-AI/genz-slang-pairs-1k", download_mode=DownloadMode.REUSE_CACHE_IF_EXISTS)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/Programmer-RD-AI/genz-slang-pairs-1k/resolve/main/README.md (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E5A4A88050>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 3c4be23e-5015-4f6a-a65f-14062b1687c4)')' thrown while requesting HEAD https://huggingface.co/datasets/Programmer-RD-AI/genz-slang-pairs-1k/resolve/main/README.md
Retrying in 1s [Retry 1/5].

KeyboardInterrupt



In [84]:
df_p = pd.DataFrame(dataset_pairs['train'])
#df_p.set_index(['normal', 'gen_z'], inplace=True)
#df_p.reset_index()
df_p.head(5)

,normal,gen_z
0,"I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some ..."
1,"I'm really tired today, I just want to relax a...","I'm hella drained today, just wanna chill at h..."
2,I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,"I'm really tired today, I think I need some co...","I'm so drained today, I gotta get me some caff..."
4,I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to tu..."


Since the data is built completely differently, whereas this exists of normal and gen z slang. We would like to produces this into the same format with the Slang, Description, Example and the generated formal and informal text form earlier.


### Extract the terms from the gen_z column
Use the LLM in the same way to create the term

In [92]:
def extract_slang(gen_z, normal_text=""):
    """Uses the local Mistral API to rewrite slang-laden text into normal English."""
    prompt = f"""You are a language refinement expert. You are extracting the slang term from the sentence.

    Requirements:
    - Capture the same emotional tone and purpose as the original slang
    - Output only UTF-8 text characters (no emojis or special symbols)
    - Provide ONLY the slang term with no explanations, quotes, or parentheses
    - Do not include any meta-commentary about the task

    Original text: {gen_z}

    Rewritten version:"""

    try:
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                "model": "mistral",
                "prompt": prompt,
                "stream": False,
                "temperature": 0.3,
                "top_p": 0.9
            },
            timeout=15
        )

        response.raise_for_status()
        data = response.json()
        result = data.get("response", "").strip()

        return result.split("\n")[0] if result else normal_text

    except requests.exceptions.JSONDecodeError:
        print("Warning: JSON parse error. Skipping this row.")
        return normal_text

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return normal_text


In [93]:
# === Apply normalization with progress bar ===
df_p["Slang"] = df_p.progress_apply(
    lambda row: extract_slang(
        row["gen_z"],
    ),
    axis=1
)

100%|██████████| 1005/1005 [03:52<00:00,  4.32it/s]


In [94]:
df_p.head(5)

,normal,gen_z,Slang
0,"I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some ...",zonked
1,"I'm really tired today, I just want to relax a...","I'm hella drained today, just wanna chill at h...",exhausted
2,I'm really excited for the concert tonight.,I'm so hype for the concert tonight.,pumped
3,"I'm really tired today, I think I need some co...","I'm so drained today, I gotta get me some caff...",burnt out need coffee
4,I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to tu...",pumped (up)


In [95]:
# Set GenZ Term to first axis
df_p.to_csv("genz-slang-pairs-1k.csv", index=False)
df_p.Slang

0                                                  zonked
1                                               exhausted
2                                                  pumped
3                                   burnt out need coffee
4                                             pumped (up)
                              ...                        
1000                             Gotta hit the sack soon.
1001                                        Fresh (phone)
1002             Chill (out) let's grab dinner post-work?
1003                                       chill (coffee)
1004    I'm really exhausted today, I kind of don't wa...
Name: Slang, Length: 1005, dtype: object